# 03 — Anomaly Detection
**Isolation Forest on Uninox Houseware demand patterns** — detect unusual spikes, stockouts, and outliers.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
from config import Paths
from knowledge.feature_store.feature_engineering import load_demand, build_sku_features
demand = load_demand()
features = build_sku_features(demand)
print(f'Feature matrix: {features.shape}')

## 1. Contamination Sweep — Choosing the Right Rate

In [ ]:
FEAT_COLS = ['net_units','lag_1','roll_mean_3','roll_std_3','is_high_season']
df = features[FEAT_COLS].dropna()
scaler = StandardScaler()
X = scaler.fit_transform(df)
results = []
for c in [0.02, 0.05, 0.08, 0.10, 0.15]:
    iso = IsolationForest(contamination=c, n_estimators=200, random_state=42)
    preds = iso.fit_predict(X)
    n_anom = (preds==-1).sum()
    results.append({'contamination':c,'anomalies':n_anom,'pct':round(n_anom/len(preds)*100,1)})
result_df = pd.DataFrame(results)
print(result_df)
fig = px.line(result_df, x='contamination', y='anomalies', markers=True,
              title='Anomaly Count vs Contamination Rate', template='plotly_dark')
fig.show()

## 2. Train Final Model (contamination=0.05)

In [ ]:
iso_final = IsolationForest(contamination=0.05, n_estimators=200, random_state=42)
iso_final.fit(X)
scores = iso_final.decision_function(X)
preds  = iso_final.predict(X)
df_plot = features.dropna(subset=FEAT_COLS).copy()
df_plot['anomaly_score'] = scores
df_plot['is_anomaly']    = (preds == -1)
print(f'Flagged: {df_plot.is_anomaly.sum()} / {len(df_plot)} rows ({df_plot.is_anomaly.mean()*100:.1f}%)')

## 3. Anomaly Score Distribution

In [ ]:
fig2 = px.histogram(df_plot, x='anomaly_score', color='is_anomaly',
                    nbins=40, barmode='overlay', template='plotly_dark',
                    title='Anomaly Score Distribution',
                    labels={'anomaly_score':'Decision Score','is_anomaly':'Is Anomaly'},
                    color_discrete_map={True:'#e74c3c', False:'#4B8BBE'})
fig2.show()

## 4. Anomalies Overlaid on SKU Time Series

In [ ]:
top_sku = demand.groupby('sku_id')['net_units'].mean().idxmax()
sku_df  = df_plot[df_plot['sku_id']==top_sku].sort_values('period')
fig3 = go.Figure()
fig3.add_scatter(x=sku_df['period'].astype(str), y=sku_df['net_units'],
                 mode='lines+markers', name='Demand', line=dict(color='#4B8BBE'))
anom = sku_df[sku_df['is_anomaly']]
fig3.add_scatter(x=anom['period'].astype(str), y=anom['net_units'],
                 mode='markers', name='Anomaly', marker=dict(color='#e74c3c', size=12, symbol='x'))
fig3.update_layout(title=f'Anomalies — {top_sku}', template='plotly_dark', xaxis_tickangle=-30)
fig3.show()

## 5. Top Anomalous Records

In [ ]:
top_anom = df_plot[df_plot['is_anomaly']].sort_values('anomaly_score').head(10)
print(top_anom[['sku_id','product_name','period','net_units','roll_mean_3','anomaly_score']].to_string())

## 6. Save Final Model

In [ ]:
from knowledge.feature_store.anomaly_model import save_anomaly_model
save_anomaly_model(iso_final, scaler)
print('Model saved to data/processed/models/anomaly_iso.pkl')